# Section 0 — Configuration
All hyperparameters and global settings. Toggle `DEBUG` for quick iteration vs full training.

In [1]:
# ========================== SECTION 0: CONFIGURATION ==========================
DEBUG = True              # True → sample 1000 rows; False → full dataset
BATCH_SIZE = 32
HIDDEN_DIM = 128
EMBEDDING_DIM = 32        # Learned node-type embedding dim
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
DROPOUT = 0.3
MAX_NODES = 512
CACHE_DIR = "./graph_cache_v2"
PATIENCE = 5
WEIGHT_DECAY = 1e-4
DEBUG_SAMPLE_SIZE = 1000
RANDOM_SEED = 42
TRUNCATION_STRATEGY = "dfs"   # "dfs" (preserves top-level) or "bfs"
USE_TOKEN_FEATURES = True     # Concat surface-level code stats after pooling
NUM_CLASSES = 2               # Task A: binary (human vs machine)

import os, pathlib
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Config: DEBUG={DEBUG}, HIDDEN={HIDDEN_DIM}, EMB={EMBEDDING_DIM}, "
      f"MAX_NODES={MAX_NODES}, TRUNC={TRUNCATION_STRATEGY}")

Config: DEBUG=True, HIDDEN=128, EMB=32, MAX_NODES=512, TRUNC=dfs


# Section 1 — Install Dependencies
Conditional installation only for missing packages.  
`torch-geometric` install uses a safe fallback chain.

In [ ]:
# ========================== SECTION 1: INSTALL DEPENDENCIES ==========================
import subprocess, sys, importlib.util

def install_if_missing(package, pip_name=None):
    """Install a pip package if its import name is not available.
    Args:
        package: Import name to check with importlib.
        pip_name: Pip package name if different from import name.
    """
    if importlib.util.find_spec(package) is None:
        name = pip_name or package
        print(f"Installing {name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
    else:
        print(f"  ✓ {package}")

# Core
for pkg, pip in [("numpy",None),("pandas",None),("sklearn","scikit-learn"),
                 ("tqdm",None),("matplotlib",None),("datasets",None)]:
    install_if_missing(pkg, pip)

install_if_missing("torch")

# Tree-sitter with version constraint
install_if_missing("tree_sitter", "tree-sitter>=0.21.0,<1.0")
for imp, pip in [("tree_sitter_python","tree-sitter-python"),
                 ("tree_sitter_cpp","tree-sitter-cpp"),
                 ("tree_sitter_java","tree-sitter-java"),
                 ("tree_sitter_javascript","tree-sitter-javascript"),
                 ("tree_sitter_go","tree-sitter-go"),
                 ("tree_sitter_c_sharp","tree-sitter-c-sharp"),
                 ("tree_sitter_c","tree-sitter-c")]:
    install_if_missing(imp, pip)

# torch-geometric: try simple install first, then find-links fallback
if importlib.util.find_spec("torch_geometric") is None:
    import torch
    print("Installing torch-geometric...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    except subprocess.CalledProcessError:
        tv = torch.__version__.split("+")[0]
        tag = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() and torch.version.cuda else "cpu"
        url = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
        print(f"  Retrying with --find-links {url}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "torch-geometric", "--find-links", url])
else:
    print("  ✓ torch_geometric")

import torch, torch_geometric
print(f"\n✓ All deps OK. torch={torch.__version__}, pyg={torch_geometric.__version__}")

# Section 2 — Imports & Setup
All imports, reproducibility seeds, and device auto-detection (CUDA → MPS → CPU).

In [ ]:
# ========================== SECTION 2: IMPORTS & SETUP ==========================
import os, sys, json, random, time, warnings, hashlib, re, platform
from collections import Counter, defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for portability
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, GlobalAttention

from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from tree_sitter import Language, Parser
import tree_sitter_python as ts_python
import tree_sitter_cpp as ts_cpp
import tree_sitter_java as ts_java
import tree_sitter_javascript as ts_javascript
import tree_sitter_go as ts_go
import tree_sitter_c_sharp as ts_c_sharp
import tree_sitter_c as ts_c

warnings.filterwarnings("ignore")

# ---- Reproducibility ----
def seed_everything(seed):
    """Set all random seeds for full reproducibility.
    Args: seed (int): Seed value.
    """
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)

# ---- Device ----
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Python : {platform.python_version()}")
print(f"PyTorch: {torch.__version__}  |  PyG: {torch_geometric.__version__}")
print(f"Device : {device}  |  OS: {platform.system()} {platform.machine()}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB)")

# Section 3 — Dataset Loading
Pure Task A focus. Loads data via Kaggle → HuggingFace → local parquet cascade.  
Includes:
- Original `id` column preservation for submission
- Near-duplicate leakage detection between train/val
- Majority-class computation for fallback predictions

In [ ]:
# ========================== SECTION 3: DATASET LOADING ==========================

def load_task_a():
    """Load Task A dataset from available sources.
    Returns: (train_df, val_df) DataFrames.
    Raises: FileNotFoundError if no source found.
    """
    # 1. Kaggle
    kaggle = "/kaggle/input/semeval-2026-task13/task_a"
    if os.path.isdir(kaggle):
        print(f"Loading from Kaggle: {kaggle}")
        return _load_parquets(kaggle)

    # 2. HuggingFace
    try:
        from datasets import load_dataset as hf_load
        print("Loading from HuggingFace: DaniilOr/SemEval-2026-Task13 (task_a)")
        ds = hf_load("DaniilOr/SemEval-2026-Task13", "task_a")
        train_df = ds["train"].to_pandas()
        val_df = ds["validation"].to_pandas() if "validation" in ds else None
        return train_df, val_df
    except Exception as e:
        print(f"  HuggingFace failed: {e}")

    # 3. Local parquets
    for base in ["./task_a", "./SemEval-2026-Task13/task_a",
                 "./SemEval-2026-Task13/SemEval-2026-Task13/task_a",
                 "../SemEval-2026-Task13/task_a"]:
        if os.path.isdir(base):
            print(f"Loading from local: {base}")
            return _load_parquets(base)

    raise FileNotFoundError("No data source found for Task A.")


def _load_parquets(base_dir):
    """Read train/val parquets from a directory.
    Args: base_dir (str): Path containing parquet files.
    Returns: (train_df, val_df) tuple.
    """
    p = pathlib.Path(base_dir)
    train_files = sorted(p.glob("*training*"))
    val_files = sorted(p.glob("*validation*"))
    train_df = pd.concat([pd.read_parquet(f) for f in train_files], ignore_index=True)
    val_df = pd.read_parquet(val_files[0]) if val_files else None
    return train_df, val_df


def check_leakage(train_df, val_df, n_chars=200):
    """Warn about near-duplicate code between train and val splits.
    Args:
        train_df, val_df: DataFrames with 'code' column.
        n_chars: Characters to hash for fingerprinting.
    """
    if val_df is None:
        return
    train_hashes = set(hashlib.md5(s[:n_chars].encode()).hexdigest()
                       for s in train_df["code"].dropna())
    val_hashes = set(hashlib.md5(s[:n_chars].encode()).hexdigest()
                     for s in val_df["code"].dropna())
    overlap = train_hashes & val_hashes
    pct = 100.0 * len(overlap) / max(len(val_hashes), 1)
    if pct > 1.0:
        print(f"⚠ LEAKAGE WARNING: {len(overlap)} val samples ({pct:.1f}%) "
              f"have near-duplicate in train (first {n_chars} chars)")
    else:
        print(f"✓ Leakage check: {pct:.2f}% overlap (OK)")


# ---- Load ----
train_df, val_df = load_task_a()

# ---- Preserve original IDs ----
if "id" not in train_df.columns:
    train_df["id"] = range(len(train_df))
if val_df is not None and "id" not in val_df.columns:
    val_df["id"] = range(len(val_df))

# ---- Clean ----
for name, df in [("train", train_df), ("val", val_df)]:
    if df is None: continue
    before = len(df)
    mask = df["code"].notna() & (df["code"].str.strip() != "")
    if name == "train": train_df = train_df[mask].reset_index(drop=True)
    else: val_df = val_df[mask].reset_index(drop=True)
    dropped = before - mask.sum()
    if dropped: print(f"  {name}: dropped {dropped} null/empty rows")

# ---- Leakage check ----
check_leakage(train_df, val_df)

# ---- Debug sampling ----
if DEBUG:
    print(f"\nDEBUG: sampling {DEBUG_SAMPLE_SIZE} rows (stratified)...")
    train_df, _ = train_test_split(train_df, train_size=min(DEBUG_SAMPLE_SIZE, len(train_df)),
                                   stratify=train_df["label"], random_state=RANDOM_SEED)
    train_df = train_df.reset_index(drop=True)
    if val_df is not None and len(val_df) > DEBUG_SAMPLE_SIZE // 2:
        val_df, _ = train_test_split(val_df, train_size=DEBUG_SAMPLE_SIZE // 2,
                                     stratify=val_df["label"], random_state=RANDOM_SEED)
        val_df = val_df.reset_index(drop=True)

# ---- Majority class (for fallback prediction) ----
MAJORITY_CLASS = int(train_df["label"].mode().iloc[0])

# ---- Stats ----
print(f"\n{'='*50}")
print(f"Task A Dataset Summary")
print(f"{'='*50}")
print(f"Train: {len(train_df):,}  |  Val: {len(val_df) if val_df is not None else 0:,}")
print(f"Majority class: {MAJORITY_CLASS}")
print(f"\nClass distribution (train):")
print(train_df["label"].value_counts().sort_index().to_string())
if "language" in train_df.columns:
    print(f"\nLanguages (train): {dict(train_df['language'].value_counts())}")
lens = train_df["code"].str.len()
print(f"Code length: mean={lens.mean():.0f}, median={lens.median():.0f}, max={lens.max():,}")

# Section 4 — AST Parser Setup
Version-safe Tree-sitter initialization with compatibility wrapper.  
Supports 7 languages: Python, C++, Java, JavaScript, Go, C#, C.

In [ ]:
# ========================== SECTION 4: AST PARSER SETUP ==========================

def _make_parser(lang_module):
    """Create a Tree-sitter parser with version-safe API detection.
    Args: lang_module: e.g. tree_sitter_python.
    Returns: Configured Parser instance.
    Raises: RuntimeError if neither API variant works.
    """
    lang_obj = Language(lang_module.language())
    # Modern API (>=0.22): Parser(language)
    try:
        p = Parser(lang_obj)
        return p
    except TypeError:
        pass
    # Older API: Parser() then set .language
    try:
        p = Parser()
        p.language = lang_obj
        return p
    except Exception:
        pass
    # Very old API: set_language
    try:
        p = Parser()
        p.set_language(lang_obj)
        return p
    except Exception as e:
        raise RuntimeError(f"Cannot initialize parser: {e}")


def build_parsers():
    """Build dict of parsers keyed by normalized language name.
    Returns: Dict[str, Parser].
    """
    modules = {"python": ts_python, "cpp": ts_cpp, "java": ts_java,
               "javascript": ts_javascript, "go": ts_go,
               "c_sharp": ts_c_sharp, "c": ts_c}
    parsers = {}
    for name, mod in modules.items():
        try:
            parsers[name] = _make_parser(mod)
        except Exception as e:
            print(f"  ⚠ {name}: {e}")
    print(f"✓ {len(parsers)} parsers initialized: {list(parsers.keys())}")
    return parsers

parsers = build_parsers()

# Language name normalization
_LANG_MAP = {"python":"python","py":"python","c++":"cpp","cpp":"cpp",
             "java":"java","javascript":"javascript","js":"javascript",
             "go":"go","golang":"go","c#":"c_sharp","csharp":"c_sharp",
             "c_sharp":"c_sharp","c":"c"}

def normalize_language(lang_str):
    """Normalize language string to parser key.
    Args: lang_str (str): Raw language name.
    Returns: Normalized key or None.
    """
    if not lang_str: return None
    return _LANG_MAP.get(lang_str.strip().lower())


def parse_code(code_str, language, parsers_dict):
    """Parse code into AST root node with error handling.
    Args:
        code_str (str): Source code.
        language (str): Language name.
        parsers_dict (dict): Parser dictionary.
    Returns: tree_sitter.Node or None on failure.
    """
    try:
        key = normalize_language(language)
        if key is None or key not in parsers_dict:
            return None
        code_bytes = code_str.encode("utf-8") if isinstance(code_str, str) else code_str
        tree = parsers_dict[key].parse(code_bytes)
        root = tree.root_node
        if root and root.child_count > 0:
            return root
        return None
    except Exception:
        return None


def ast_stats(node):
    """Compute depth and node count of AST.
    Args: node: tree_sitter.Node.
    Returns: (depth, count) tuple.
    """
    stack = [(node, 0)]
    max_d, cnt = 0, 0
    while stack:
        n, d = stack.pop()
        cnt += 1
        if d > max_d: max_d = d
        for c in n.children:
            stack.append((c, d + 1))
    return max_d, cnt

### Smoke Test — AST Parsing

In [ ]:
# ---- SMOKE TEST: AST Parsing ----
print("AST Parsing Smoke Test\n" + "-"*50)
test_langs = train_df["language"].dropna().unique() if "language" in train_df.columns else ["python"]
for lang in test_langs:
    sub = train_df[train_df["language"]==lang] if "language" in train_df.columns else train_df
    if len(sub) == 0: continue
    root = parse_code(sub.iloc[0]["code"], lang, parsers)
    if root:
        depth, count = ast_stats(root)
        print(f"  ✓ {lang:12s} → depth={depth:3d}, nodes={count:5d}")
        assert depth > 0 and count > 1
    else:
        nl = normalize_language(lang)
        status = "parse failed" if nl and nl in parsers else "unsupported"
        print(f"  ✗ {lang:12s} → {status}")
print("\n✓ Smoke test complete")

# Section 5 — AST to Graph Conversion

Major redesign from v1:
- **DFS traversal** (default) preserves top-level structure during truncation
- **`collections.deque`** for O(1) popleft instead of O(n) `list.pop(0)`
- **Richer edges**: parent↔child + sibling + sequential (next-token)
- **Node features split**: `type_ids` (LongTensor for nn.Embedding) + `cont_features` (FloatTensor)
- **New features**: subtree size (normalized), position-in-parent (normalized)
- **Optional token-level features**: avg identifier length, comment ratio, indentation variance, unique operators

In [ ]:
# ========================== SECTION 5: AST TO GRAPH CONVERSION ==========================

def build_node_vocab(df, parsers_dict, max_samples=None):
    """Scan training data to build vocabulary of AST node types.
    Args:
        df: Training DataFrame.
        parsers_dict: Parser dict.
        max_samples: Cap on samples to scan (None=all).
    Returns: Dict[str, int] mapping node type to index. Index 0 = <UNK>.
    """
    node_types = set()
    n = len(df) if max_samples is None else min(max_samples, len(df))
    parsed = 0
    for idx in tqdm(range(n), desc="Building node vocab"):
        row = df.iloc[idx]
        root = parse_code(row["code"], row.get("language", "python"), parsers_dict)
        if root is None:
            continue
        stack = [root]
        while stack:
            nd = stack.pop()
            node_types.add(nd.type)
            stack.extend(nd.children)
        parsed += 1
    vocab = {"<UNK>": 0}
    for i, t in enumerate(sorted(node_types), 1):
        vocab[t] = i
    print(f"✓ Node vocab: {len(vocab)} types from {parsed}/{n} parsed samples")
    return vocab


def _subtree_sizes(node, cache=None):
    """Compute subtree size for every node (memoized).
    Args: node: tree_sitter.Node, cache: optional dict.
    Returns: int subtree size of this node.
    """
    if cache is None:
        cache = {}
    nid = id(node)
    if nid in cache:
        return cache[nid]
    s = 1
    for c in node.children:
        s += _subtree_sizes(c, cache)
    cache[nid] = s
    return s


def ast_to_graph(root_node, node_vocab, max_nodes=512, strategy="dfs"):
    """Convert AST to graph with rich edges and split features.

    Args:
        root_node: tree_sitter.Node root.
        node_vocab: Node type vocabulary dict.
        max_nodes: Truncation limit.
        strategy: 'dfs' (preserves top-level) or 'bfs'.

    Returns:
        Dict with keys: 'type_ids' (LongTensor[N]),
        'cont_features' (FloatTensor[N, 6]), 'edge_index' (LongTensor[2, E]),
        'num_nodes' (int). Or None if graph is empty.
    """
    # ---- Collect nodes via DFS or BFS ----
    # Each entry: (node, parent_idx, depth, position_in_parent)
    collected = []  # (node, depth, parent_idx, pos_in_parent)
    size_cache = {}
    _subtree_sizes(root_node, size_cache)

    if strategy == "dfs":
        stack = deque([(root_node, 0, -1, 0)])  # node, depth, parent_idx, pos
        while stack and len(collected) < max_nodes:
            node, depth, pidx, pos = stack.pop()  # LIFO = DFS
            cur = len(collected)
            collected.append((node, depth, pidx, pos))
            # Push children in reverse so first child is visited first
            children = list(node.children)
            for ci in range(len(children) - 1, -1, -1):
                if len(collected) + len(stack) < max_nodes:
                    stack.append((children[ci], depth + 1, cur, ci))
    else:  # BFS
        queue = deque([(root_node, 0, -1, 0)])
        while queue and len(collected) < max_nodes:
            node, depth, pidx, pos = queue.popleft()
            cur = len(collected)
            collected.append((node, depth, pidx, pos))
            for ci, child in enumerate(node.children):
                if len(collected) + len(queue) < max_nodes:
                    queue.append((child, depth + 1, cur, ci))

    if len(collected) == 0:
        return None

    n = len(collected)

    # ---- Edges ----
    src, dst = [], []

    # Track children per parent for sibling edges
    parent_children = defaultdict(list)

    for cur, (node, depth, pidx, pos) in enumerate(collected):
        # Parent ↔ child (bidirectional)
        if pidx >= 0:
            src.extend([pidx, cur])
            dst.extend([cur, pidx])
        parent_children[pidx].append(cur)

    # Sibling edges: connect consecutive children of same parent
    for pidx, children_ids in parent_children.items():
        if pidx < 0:
            continue
        for i in range(len(children_ids) - 1):
            a, b = children_ids[i], children_ids[i + 1]
            src.extend([a, b])
            dst.extend([b, a])

    # Sequential (next-token) edges: connect nodes in traversal order
    for i in range(n - 1):
        src.extend([i, i + 1])
        dst.extend([i + 1, i])

    # ---- Features ----
    max_depth = max(d for _, d, _, _ in collected) or 1
    max_children = max(nd.child_count for nd, _, _, _ in collected) or 1
    total_nodes_in_tree = size_cache.get(id(root_node), n)

    type_ids = []
    cont_features = []

    for node, depth, pidx, pos in collected:
        # Node type index (integer — will be embedded by nn.Embedding)
        type_ids.append(node_vocab.get(node.type, 0))

        # Continuous features [6 dims]
        parent_nchildren = collected[pidx][0].child_count if pidx >= 0 else 1
        feat = [
            depth / max_depth,                                       # normalized depth
            node.child_count / max_children,                        # normalized child count
            1.0 if node.child_count == 0 else 0.0,                  # is_leaf
            1.0 if node.is_named else 0.0,                          # is_named
            size_cache.get(id(node), 1) / total_nodes_in_tree,      # subtree size (norm)
            pos / max(parent_nchildren - 1, 1),                     # position in parent (norm)
        ]
        cont_features.append(feat)

    if len(src) > 0:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
    else:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)

    return {
        "type_ids": torch.tensor(type_ids, dtype=torch.long),
        "cont_features": torch.tensor(cont_features, dtype=torch.float),
        "edge_index": edge_index,
        "num_nodes": n,
    }


# ---- Token-level feature extraction ----
def extract_token_features(code_str):
    """Extract surface-level code statistics as a feature vector.
    Args: code_str (str): Source code.
    Returns: List of 4 floats: [avg_id_len, comment_ratio, indent_var, unique_ops].
    """
    lines = code_str.split("\n")
    n_lines = max(len(lines), 1)

    # Avg identifier length
    identifiers = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', code_str)
    avg_id_len = np.mean([len(x) for x in identifiers]) if identifiers else 0.0

    # Comment ratio (simple heuristic: lines starting with // or #)
    comment_lines = sum(1 for l in lines if l.strip().startswith(("//", "#", "/*", "*")))
    comment_ratio = comment_lines / n_lines

    # Indentation variance
    indents = [len(l) - len(l.lstrip()) for l in lines if l.strip()]
    indent_var = np.var(indents) if len(indents) > 1 else 0.0
    indent_var = min(indent_var / 100.0, 1.0)  # normalize

    # Unique operator count (normalized)
    ops = re.findall(r'[+\-*/%=<>!&|^~]+', code_str)
    unique_ops = len(set(ops))
    unique_ops = min(unique_ops / 20.0, 1.0)  # normalize

    return [avg_id_len / 20.0, comment_ratio, indent_var, unique_ops]

TOKEN_FEATURE_DIM = 4


# ---- Build vocab ----
vocab_n = len(train_df) if not DEBUG else min(500, len(train_df))
node_vocab = build_node_vocab(train_df, parsers, max_samples=vocab_n)
VOCAB_SIZE = len(node_vocab)
CONT_FEAT_DIM = 6
print(f"VOCAB_SIZE={VOCAB_SIZE}, CONT_FEAT_DIM={CONT_FEAT_DIM}, "
      f"TOKEN_FEAT_DIM={TOKEN_FEATURE_DIM}")

### Smoke Test — AST to Graph

In [ ]:
# ---- SMOKE TEST: AST → Graph ----
print("AST → Graph Smoke Test (v2)\n" + "-"*50)
for i in range(min(5, len(train_df))):
    row = train_df.iloc[i]
    root = parse_code(row["code"], row.get("language","python"), parsers)
    if root is None:
        print(f"  Sample {i}: parse failed — skip"); continue
    g = ast_to_graph(root, node_vocab, MAX_NODES, TRUNCATION_STRATEGY)
    if g is None:
        print(f"  Sample {i}: empty graph — skip"); continue

    assert g["type_ids"].ndim == 1, f"type_ids should be 1D, got {g['type_ids'].shape}"
    assert g["cont_features"].shape == (g["num_nodes"], CONT_FEAT_DIM)
    assert g["edge_index"].shape[0] == 2
    assert g["num_nodes"] <= MAX_NODES

    tf = extract_token_features(row["code"])
    assert len(tf) == TOKEN_FEATURE_DIM

    print(f"  Sample {i}: nodes={g['num_nodes']}, edges={g['edge_index'].shape[1]}, "
          f"type_ids={g['type_ids'][:5].tolist()}..., token_feat={[f'{x:.2f}' for x in tf]}")

print("\n✓ Smoke test passed")

# Section 6 — Graph Dataset Class

Proper `torch.utils.data.Dataset` subclass with:
- Lazy `torch.load()` in `__getitem__` — **no `to_list()` call**
- Disk caching of processed graphs as `.pt` files
- Parse-failure skip logging
- Token features computed and stored alongside graph data

In [ ]:
# ========================== SECTION 6: GRAPH DATASET CLASS ==========================

class CodeGraphDataset(torch.utils.data.Dataset):
    """Lazy-loading graph dataset with disk caching.

    Args:
        df: DataFrame with 'code', 'label', 'language' columns.
        parsers_dict: Tree-sitter parsers dict.
        node_vocab: Node type vocabulary.
        cache_dir: Directory for .pt file cache.
        max_nodes: Graph truncation limit.
        split_name: Cache subdirectory name.
        use_token_features: Whether to compute token-level features.
        strategy: Truncation strategy ('dfs' or 'bfs').
    """
    def __init__(self, df, parsers_dict, node_vocab, cache_dir,
                 max_nodes=512, split_name="data",
                 use_token_features=True, strategy="dfs"):
        super().__init__()
        self.cache_path = os.path.join(cache_dir, split_name)
        os.makedirs(self.cache_path, exist_ok=True)
        self.use_token_features = use_token_features
        self.valid_indices = []
        self.skip_count = 0

        self._process(df, parsers_dict, node_vocab, max_nodes, strategy)

    def _cache_file(self, idx):
        return os.path.join(self.cache_path, f"g_{idx}.pt")

    def _process(self, df, parsers_dict, node_vocab, max_nodes, strategy):
        """Convert code samples to graphs and cache to disk."""
        print(f"  Processing {len(df)} samples → {self.cache_path}")
        for idx in tqdm(range(len(df)), desc=f"  Graphs", leave=True):
            cf = self._cache_file(idx)
            if os.path.exists(cf):
                self.valid_indices.append(idx)
                continue

            row = df.iloc[idx]
            root = parse_code(row["code"], row.get("language", "python"), parsers_dict)
            if root is None:
                self.skip_count += 1
                continue

            g = ast_to_graph(root, node_vocab, max_nodes, strategy)
            if g is None or g["num_nodes"] == 0:
                self.skip_count += 1
                continue

            data = Data(
                x_type=g["type_ids"],           # LongTensor[N] for embedding
                x_cont=g["cont_features"],       # FloatTensor[N, 6]
                edge_index=g["edge_index"],
                y=torch.tensor(int(row["label"]), dtype=torch.long),
                num_nodes=g["num_nodes"],
            )

            if self.use_token_features:
                tf = extract_token_features(row["code"])
                data.token_features = torch.tensor(tf, dtype=torch.float)

            torch.save(data, cf)
            self.valid_indices.append(idx)

        print(f"  ✓ {len(self.valid_indices)} graphs, {self.skip_count} skipped")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        """Lazy load a single graph from disk."""
        real_idx = self.valid_indices[idx]
        return torch.load(self._cache_file(real_idx), weights_only=False)


# ---- Build datasets ----
print("Building train dataset...")
train_dataset = CodeGraphDataset(
    train_df, parsers, node_vocab, CACHE_DIR,
    max_nodes=MAX_NODES, split_name="task_a_train_v2",
    use_token_features=USE_TOKEN_FEATURES, strategy=TRUNCATION_STRATEGY)

val_dataset = None
if val_df is not None:
    print("Building val dataset...")
    val_dataset = CodeGraphDataset(
        val_df, parsers, node_vocab, CACHE_DIR,
        max_nodes=MAX_NODES, split_name="task_a_val_v2",
        use_token_features=USE_TOKEN_FEATURES, strategy=TRUNCATION_STRATEGY)

print(f"\nTrain: {len(train_dataset)} graphs  |  "
      f"Val: {len(val_dataset) if val_dataset else 0} graphs")

### Smoke Test — Graph Dataset

In [ ]:
# ---- SMOKE TEST: Graph Dataset ----
print("Graph Dataset Smoke Test\n" + "-"*50)
assert len(train_dataset) > 0, "Dataset empty — all samples failed"

s = train_dataset[0]
assert hasattr(s, "x_type"), "Missing x_type (node type indices)"
assert hasattr(s, "x_cont"), "Missing x_cont (continuous features)"
assert hasattr(s, "edge_index"), "Missing edge_index"
assert hasattr(s, "y"), "Missing y (label)"
assert s.x_type.dtype == torch.long
assert s.x_cont.shape[1] == CONT_FEAT_DIM
if USE_TOKEN_FEATURES:
    assert hasattr(s, "token_features"), "Missing token_features"
    assert s.token_features.shape[0] == TOKEN_FEATURE_DIM

print(f"  Sample 0: x_type={s.x_type.shape}, x_cont={s.x_cont.shape}, "
      f"edges={s.edge_index.shape}, label={s.y.item()}")

# Test DataLoader batching
test_loader = PyGDataLoader(train_dataset, batch_size=4, shuffle=False)
batch = next(iter(test_loader))
print(f"  Batch:  x_type={batch.x_type.shape}, x_cont={batch.x_cont.shape}, "
      f"y={batch.y.shape}, batch_vec={batch.batch.shape}")
print("\n✓ Smoke test passed")

# Section 7 — GNN Model

Architecture overhaul from v1:
- **`nn.Embedding`** for node types (categorical, not ordinal)
- **3 GCN layers** with **residual connections**
- **GAT layer** (4 heads, concat=False)
- **`GlobalAttention`** pooling (learned, not plain mean)
- **Optional token feature concatenation** after pooling
- Deeper MLP classifier head

In [ ]:
# ========================== SECTION 7: GNN MODEL ==========================

class CodeGNN(nn.Module):
    """Graph Neural Network with learned node embeddings and attention pooling.

    Args:
        vocab_size: Number of node types (for embedding).
        embed_dim: Node type embedding dimension.
        cont_dim: Continuous feature dimension per node.
        hidden_dim: Hidden layer dimension.
        num_classes: Output classes.
        dropout: Dropout rate.
        use_token_features: Whether to expect token-level features.
        token_feat_dim: Dimension of token features.
    """
    def __init__(self, vocab_size, embed_dim, cont_dim, hidden_dim,
                 num_classes, dropout=0.3, use_token_features=True,
                 token_feat_dim=4):
        super().__init__()
        self.use_token_features = use_token_features

        # Node type embedding (categorical → dense)
        self.type_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        input_dim = embed_dim + cont_dim

        # Input projection
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)

        # 3× GCN with residual connections
        self.gcn1 = GCNConv(hidden_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.gcn3 = GCNConv(hidden_dim, hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)

        # GAT layer
        self.gat = GATConv(hidden_dim, hidden_dim, heads=4, concat=False)
        self.norm_gat = nn.LayerNorm(hidden_dim)

        # Attention pooling
        gate_nn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2),
                                nn.ReLU(), nn.Linear(hidden_dim // 2, 1))
        self.pool = GlobalAttention(gate_nn)

        # Classifier MLP
        mlp_in = hidden_dim + (token_feat_dim if use_token_features else 0)
        self.fc1 = nn.Linear(mlp_in, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_type, x_cont, edge_index, batch, token_features=None):
        """Forward pass.
        Args:
            x_type: LongTensor[N] node type indices.
            x_cont: FloatTensor[N, cont_dim] continuous node features.
            edge_index: LongTensor[2, E] edge indices.
            batch: LongTensor[N] batch assignment vector.
            token_features: Optional FloatTensor[B, token_feat_dim].
        Returns: FloatTensor[B, num_classes] logits.
        """
        # Embed node types + concat continuous features
        te = self.type_embed(x_type)                    # [N, embed_dim]
        h = torch.cat([te, x_cont], dim=1)              # [N, embed+cont]
        h = F.relu(self.input_norm(self.input_proj(h)))  # [N, hidden]

        # GCN block 1 with residual
        res = h
        h = F.relu(self.norm1(self.gcn1(h, edge_index)))
        h = self.dropout(h) + res                        # residual

        # GCN block 2 with residual
        res = h
        h = F.relu(self.norm2(self.gcn2(h, edge_index)))
        h = self.dropout(h) + res

        # GCN block 3 with residual
        res = h
        h = F.relu(self.norm3(self.gcn3(h, edge_index)))
        h = self.dropout(h) + res

        # GAT
        h = F.relu(self.norm_gat(self.gat(h, edge_index)))

        # Attention pooling → [B, hidden]
        h = self.pool(h, batch)

        # Concat token features if available
        if self.use_token_features and token_features is not None:
            h = torch.cat([h, token_features], dim=1)    # [B, hidden+token]

        # MLP classifier
        h = F.relu(self.fc1(h))
        h = self.dropout(h)
        h = self.fc2(h)  # [B, num_classes] raw logits
        return h


# ---- Instantiate ----
model = CodeGNN(
    vocab_size=VOCAB_SIZE, embed_dim=EMBEDDING_DIM,
    cont_dim=CONT_FEAT_DIM, hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES, dropout=DROPOUT,
    use_token_features=USE_TOKEN_FEATURES,
    token_feat_dim=TOKEN_FEATURE_DIM
).to(device)

total_p = sum(p.numel() for p in model.parameters())
print(f"CodeGNN: {total_p:,} parameters")
print(model)

### Smoke Test — GNN Model

In [ ]:
# ---- SMOKE TEST: GNN Model ----
print("GNN Model Smoke Test\n" + "-"*50)
test_loader = PyGDataLoader(train_dataset, batch_size=min(8, len(train_dataset)), shuffle=False)
batch = next(iter(test_loader)).to(device)

with torch.no_grad():
    tf = batch.token_features if USE_TOKEN_FEATURES and hasattr(batch, "token_features") else None
    out = model(batch.x_type, batch.x_cont, batch.edge_index, batch.batch, tf)

B = batch.y.shape[0]
expected = (B, NUM_CLASSES)
assert out.shape == expected, f"Expected {expected}, got {out.shape}"
print(f"  Input:  x_type={batch.x_type.shape}, x_cont={batch.x_cont.shape}")
print(f"  Output: {out.shape} ✓")
print(f"  Logits: {out[0].cpu().numpy()}")
print("\n✓ Smoke test passed")

# Section 8 — Training Pipeline
- DataLoader wraps the Dataset **directly** (no `to_list()`)
- Class-weighted CrossEntropyLoss
- Early stopping on validation macro F1
- Gradient clipping (max_norm=1.0)

In [ ]:
# ========================== SECTION 8: TRAINING PIPELINE ==========================

def compute_class_weights(dataset):
    """Compute inverse-frequency class weights.
    Args: dataset: CodeGraphDataset.
    Returns: FloatTensor of per-class weights on device.
    """
    labels = []
    for i in range(len(dataset)):
        labels.append(dataset[i].y.item())
    counts = Counter(labels)
    total = sum(counts.values())
    w = []
    for c in range(NUM_CLASSES):
        cnt = counts.get(c, 1)
        w.append(total / (NUM_CLASSES * cnt))
    w = torch.tensor(w, dtype=torch.float)
    w = w / w.sum() * NUM_CLASSES  # normalize mean=1
    return w.to(device)


def _forward_batch(model, batch, dev):
    """Run forward pass handling token features.
    Args: model, batch (PyG Batch), dev (device).
    Returns: logits tensor.
    """
    batch = batch.to(dev)
    tf = None
    if USE_TOKEN_FEATURES and hasattr(batch, "token_features"):
        tf = batch.token_features
    return model(batch.x_type, batch.x_cont, batch.edge_index, batch.batch, tf)


def train_epoch(model, loader, optimizer, criterion, dev):
    """Train one epoch.
    Args: model, loader, optimizer, criterion, dev.
    Returns: Dict with 'loss' and 'f1'.
    """
    model.train()
    total_loss, all_p, all_t = 0, [], []
    for batch in tqdm(loader, desc="  Train", leave=False):
        optimizer.zero_grad()
        out = _forward_batch(model, batch, dev)
        loss = criterion(out, batch.to(dev).y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.y.size(0)
        all_p.extend(out.argmax(1).cpu().numpy())
        all_t.extend(batch.y.cpu().numpy())
    return {"loss": total_loss / len(all_t),
            "f1": f1_score(all_t, all_p, average="macro", zero_division=0)}


def eval_epoch(model, loader, criterion, dev):
    """Evaluate on a dataset.
    Args: model, loader, criterion, dev.
    Returns: Dict with 'loss', 'f1', 'preds', 'labels'.
    """
    model.eval()
    total_loss, all_p, all_t = 0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Eval ", leave=False):
            out = _forward_batch(model, batch, dev)
            loss = criterion(out, batch.to(dev).y)
            total_loss += loss.item() * batch.y.size(0)
            all_p.extend(out.argmax(1).cpu().numpy())
            all_t.extend(batch.y.cpu().numpy())
    return {"loss": total_loss / max(len(all_t), 1),
            "f1": f1_score(all_t, all_p, average="macro", zero_division=0),
            "preds": all_p, "labels": all_t}


# ---- DataLoaders (wrap dataset directly — NO to_list) ----
train_loader = PyGDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = PyGDataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False) if val_dataset else None

# ---- Loss + optimizer ----
class_weights = compute_class_weights(train_dataset)
print(f"Class weights: {class_weights.cpu().numpy()}")
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ---- Training loop ----
best_f1, patience_ctr = 0.0, 0
history = {"train_loss":[], "val_loss":[], "train_f1":[], "val_f1":[]}

print(f"\nTraining for up to {NUM_EPOCHS} epochs (patience={PATIENCE})...")
print(f"{'Ep':>3} | {'TrLoss':>8} | {'VaLoss':>8} | {'TrF1':>6} | {'VaF1':>6} | {'Time':>5}")
print("-" * 52)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch(model, train_loader, optimizer, criterion, device)
    history["train_loss"].append(tr["loss"])
    history["train_f1"].append(tr["f1"])

    if val_loader:
        va = eval_epoch(model, val_loader, criterion, device)
        history["val_loss"].append(va["loss"])
        history["val_f1"].append(va["f1"])
        cur_f1 = va["f1"]
        vl_str, vf_str = f'{va["loss"]:.4f}', f'{va["f1"]:.4f}'
    else:
        cur_f1 = tr["f1"]
        history["val_loss"].append(0); history["val_f1"].append(0)
        vl_str = vf_str = "  N/A "

    dt = time.time() - t0
    print(f"{epoch:3d} | {tr['loss']:8.4f} | {vl_str:>8} | "
          f"{tr['f1']:6.4f} | {vf_str:>6} | {dt:4.1f}s", end="")

    if cur_f1 > best_f1:
        best_f1 = cur_f1
        patience_ctr = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(" ★")
    else:
        patience_ctr += 1
        print()
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stop at epoch {epoch}")
            break

print(f"\n✓ Best F1: {best_f1:.4f}")

# Section 9 — Evaluation
Load best checkpoint. Report metrics, confusion matrix heatmap, and error analysis with original IDs.

In [ ]:
# ========================== SECTION 9: EVALUATION ==========================
model.load_state_dict(torch.load("best_model.pt", map_location=device, weights_only=True))
model.eval()
print("✓ Best model loaded")

if val_loader:
    results = eval_epoch(model, val_loader, criterion, device)
    eval_label = "Validation"
else:
    results = eval_epoch(model, train_loader, criterion, device)
    eval_label = "Training (no val set)"

y_true, y_pred = results["labels"], results["preds"]

# ---- Metrics ----
print(f"\n{'='*50}")
print(f"Evaluation on {eval_label} — Task A")
print(f"{'='*50}")
print(f"Macro F1  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

class_names = {0: "Human", 1: "Machine"}
names = [class_names.get(i, str(i)) for i in range(NUM_CLASSES)]
print(f"\n{classification_report(y_true, y_pred, target_names=names, zero_division=0)}")

# ---- Confusion Matrix ----
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_title("Confusion Matrix — Task A", fontsize=13)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(names)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        c = "white" if cm[i,j] > cm.max()/2 else "black"
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", color=c, fontsize=12)
plt.tight_layout()
plt.savefig("confusion_matrix_task_A.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved confusion_matrix_task_A.png")

In [ ]:
# ---- Error Analysis ----
print(f"\n{'='*50}")
print("Error Analysis — 5 Misclassified Examples Per Class")
print(f"{'='*50}")

eval_df = val_df if val_df is not None else train_df
eval_ds = val_dataset if val_dataset is not None else train_dataset

errors_by_class = defaultdict(list)
for i, (t, p) in enumerate(zip(y_true, y_pred)):
    if t != p:
        errors_by_class[t].append((i, p))

for cls in range(NUM_CLASSES):
    cname = class_names.get(cls, str(cls))
    errs = errors_by_class.get(cls, [])
    print(f"\n--- {cname} (class {cls}): {len(errs)} errors ---")
    for idx, pred in errs[:5]:
        if idx < len(eval_ds.valid_indices):
            orig = eval_ds.valid_indices[idx]
            if orig < len(eval_df):
                row = eval_df.iloc[orig]
                snippet = row["code"][:120].replace("\n", " ")
                orig_id = row.get("id", orig)
                pname = class_names.get(pred, str(pred))
                print(f"  id={orig_id} | True={cname} Pred={pname} | {snippet}...")

# Section 10 — Inference & Submission

- `predict()` uses **majority class** fallback (not hardcoded 0)
- `generate_submission()` preserves **original `id` column** from dataset
- Format validation against label mapping

In [ ]:
# ========================== SECTION 10: INFERENCE & SUBMISSION ==========================

def predict(model, code_str, language, parsers_dict, node_vocab_dict, dev,
            fallback_label=None):
    """Predict label for a single code snippet.
    Args:
        model: Trained CodeGNN.
        code_str (str): Source code.
        language (str): Language name.
        parsers_dict: Parser dict.
        node_vocab_dict: Node vocab.
        dev: torch.device.
        fallback_label (int): Label to return on parse failure.
    Returns: int predicted label ID.
    """
    if fallback_label is None:
        fallback_label = MAJORITY_CLASS

    model.eval()
    root = parse_code(code_str, language, parsers_dict)
    if root is None:
        return fallback_label

    g = ast_to_graph(root, node_vocab_dict, MAX_NODES, TRUNCATION_STRATEGY)
    if g is None or g["num_nodes"] == 0:
        return fallback_label

    data = Data(
        x_type=g["type_ids"],
        x_cont=g["cont_features"],
        edge_index=g["edge_index"],
        num_nodes=g["num_nodes"],
    )
    data.batch = torch.zeros(g["num_nodes"], dtype=torch.long)

    tf = None
    if USE_TOKEN_FEATURES:
        tf_vec = extract_token_features(code_str)
        tf = torch.tensor([tf_vec], dtype=torch.float).to(dev)

    data = data.to(dev)
    with torch.no_grad():
        logits = model(data.x_type, data.x_cont, data.edge_index, data.batch, tf)
    return logits.argmax(1).item()


def generate_submission(model, test_df, parsers_dict, node_vocab_dict, dev):
    """Generate submission CSV preserving original IDs.
    Args:
        model: Trained CodeGNN.
        test_df: DataFrame with 'code', 'language', and optionally 'id'.
        parsers_dict, node_vocab_dict: As usual.
        dev: torch.device.
    Returns: DataFrame with 'id' and 'label' columns.
    """
    model.eval()
    rows = []
    for idx in tqdm(range(len(test_df)), desc="Predicting"):
        row = test_df.iloc[idx]
        pred = predict(model, row["code"], row.get("language", "python"),
                       parsers_dict, node_vocab_dict, dev)
        # Use original id if available, else row index
        sample_id = row["id"] if "id" in row.index else idx
        rows.append({"id": int(sample_id), "label": int(pred)})
    return pd.DataFrame(rows)


# ---- Demo prediction ----
demo = train_df.iloc[0]
dp = predict(model, demo["code"], demo.get("language","python"), parsers, node_vocab, device)
print(f"Demo: true={demo['label']}, predicted={dp}")

# ---- Generate submission ----
sub_src = val_df if val_df is not None else train_df
print(f"\nGenerating submission on {'val' if val_df is not None else 'train'} set...")
submission = generate_submission(model, sub_src, parsers, node_vocab, device)

# ---- Validate ----
assert list(submission.columns) == ["id", "label"]
assert submission["label"].dtype in [np.int64, np.int32, int]
invalid = set(submission["label"].unique()) - set(range(NUM_CLASSES))
assert not invalid, f"Invalid labels: {invalid}"

out_path = "submission_task_A.csv"
submission.to_csv(out_path, index=False)
print(f"\n✓ Saved {out_path}")
print(f"  Rows: {len(submission):,}")
print(f"  Labels: {dict(submission['label'].value_counts().sort_index())}")
print(submission.head())

### Training Curves

In [ ]:
# ---- Training curves ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(history["train_loss"], "o-", ms=3, label="Train")
if any(v > 0 for v in history["val_loss"]):
    ax1.plot(history["val_loss"], "s-", ms=3, label="Val")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Loss Curves")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history["train_f1"], "o-", ms=3, label="Train")
if any(v > 0 for v in history["val_f1"]):
    ax2.plot(history["val_f1"], "s-", ms=3, label="Val")
ax2.set(xlabel="Epoch", ylabel="Macro F1", title="Macro F1 Curves")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves_task_A.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved training_curves_task_A.png")